WEEK 6 ASSIGNMENT
DATASET USED = SAMPLE SUPERSTORE

In [1]:
!pip install pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week6_Spark") \
    .getOrCreate()

Q1
1. Driver Node

Role: The brain of the Spark application.



Starts the application and creates the SparkSession.
Converts and optimizes user code into an execution plan (DAG).
Splits jobs into tasks and assigns them to executors.
Tracks task progress and collects final results.

2. Cluster Manager

Role: Manages the cluster's resources.



Allocates CPU and memory across machines.
Launches executors requested by the Driver.
Monitors node health and manages resources for multiple applications.
3. Executor

Role: The worker that performs the actual processing.



Executes tasks assigned by the Driver.
Processes data and sends results back.
Stores cached data in memory and writes shuffle data to disk when needed.

Q2
1. Catalyst Optimizer (Blueprint Creation)

Spark uses lazy evaluation, meaning it doesn't execute transformations immediately. Instead, it builds a logical plan (DAG) of all operations and waits until an action (like .count(), .show(), or .write()) is called.

2. Performance Benefits
Predicate Pushdown: Applies filters while reading data, reducing disk I/O and memory usage.
Column Pruning: Reads only the columns actually needed instead of the entire dataset.
IT ALSO Combines multiple operations into a single execution stage, avoiding unnecessary intermediate data.
Redundant Calculation Removal: Skips transformations that are no longer needed before execution, improving efficiency.

In [2]:
#Q3

df_superstore = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("Sample - Superstore.csv")


df_superstore.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



Q4
CSV vs. Parquet: Storage & Performance
1. Storage Format
CSV (Row-Based): Stores data row by row as plain text. It is human-readable but has no built-in schema or compression.
Parquet (Columnar): Stores data column by column in a compressed binary format with embedded schema and data types.
2. Why Parquet is Faster
Column Pruning: Reads only the required columns instead of the entire dataset, reducing disk I/O.
Predicate Pushdown: Uses metadata to skip irrelevant data blocks when filtering.
NOW Similar data stored together compresses much more efficiently, saving storage and improving read speed.
No Schema Inference: Data types are already stored in the file, so Spark can read Parquet faster and more accurately than CSV.

In [7]:
#Q5
df_electronics = df_superstore.filter(df_superstore["Category"] == "Technology") \
                               .select("Product ID", "Sales")

df_electronics.show(5)

+---------------+--------+
|     Product ID|   Sales|
+---------------+--------+
|TEC-PH-10002275| 907.152|
|TEC-PH-10002033| 911.424|
|TEC-PH-10001949|  213.48|
|TEC-AC-10003027|   90.57|
|TEC-PH-10004977|1097.544|
+---------------+--------+
only showing top 5 rows


In [8]:
#Q6

df_superstore_revised = df_superstore.withColumnRenamed("Product Name", "item_name") \
                                     .withColumn("Sales", df_superstore["Sales"].cast("double"))

df_superstore_revised.select("item_name", "Sales").show(5)

+--------------------+--------+
|           item_name|   Sales|
+--------------------+--------+
|Bush Somerset Col...|  261.96|
|Hon Deluxe Fabric...|  731.94|
|Self-Adhesive Add...|   14.62|
|Bretford CR4500 S...|957.5775|
|Eldon Fold 'N Rol...|  22.368|
+--------------------+--------+
only showing top 5 rows


Q7
1. What is a Lineage Graph?

A Lineage Graph (DAG) records all transformations (like .filter(), .map(), and .select()) applied to a DataFrame. Spark builds this graph instead of executing operations immediately, enabling lazy evaluation and recovery from failures.

2. How Fault Tolerance Works
Partition-Based Recovery: Data is divided into partitions across worker nodes.
Failure Detection: If a node fails, only the partitions on that node are lost.
Recomputation: Spark uses the Lineage Graph to recreate only the missing partitions instead of rerunning the entire job.
Task Rescheduling: The Driver assigns these recomputation tasks to healthy executors, ensuring reliable execution.

In [12]:
#Q8
df_superstore = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .csv("Sample - Superstore.csv")

from pyspark.sql.functions import col

df_superstore_filtered = df_superstore \
    .withColumn("Sales", col("Sales").cast("double")) \
    .filter((col("Ship Mode") == "Second Class") & (col("Sales") > 1000))

df_superstore_filtered.select("Order ID", "Ship Mode", "Sales").show(5)

+--------------+------------+--------+
|      Order ID|   Ship Mode|   Sales|
+--------------+------------+--------+
|CA-2014-131926|Second Class| 2001.86|
|CA-2014-131926|Second Class| 1503.25|
|US-2014-106992|Second Class|3059.982|
|US-2014-106992|Second Class|2519.958|
|US-2015-161991|Second Class|  1114.4|
+--------------+------------+--------+
only showing top 5 rows


Q9
1. What is Predicate Pushdown?

Predicate Pushdown is an optimization where Spark applies filter conditions (like WHERE or .filter()) at the storage level before loading data into memory, so only relevant data is read.

2. How Parquet Supports It

Parquet stores metadata (such as minimum, maximum, and null counts) for each row group. Spark checks this metadata before reading data and skips row groups that cannot satisfy the filter condition.

3. Performance Benefits
Less Disk I/O:There Reads only the required data blocks.
Lower Memory Usage: Unnecessary data is never loaded into executor memory.
Faster Queries: Processing less data improves execution speed and helps avoid OutOfMemory (OOM) errors.

In [13]:
#Q10
from pyspark.sql.functions import col, round

df_superstore_taxed = df_superstore \
    .withColumn("Sales", col("Sales").cast("double")) \
    .withColumn("final_price", round(col("Sales") * 1.18, 2))

df_superstore_taxed.select("Order ID", "Sales", "final_price").show(5)

+--------------+--------+-----------+
|      Order ID|   Sales|final_price|
+--------------+--------+-----------+
|CA-2016-152156|  261.96|     309.11|
|CA-2016-152156|  731.94|     863.69|
|CA-2016-138688|   14.62|      17.25|
|US-2015-108966|957.5775|    1129.94|
|US-2015-108966|  22.368|      26.39|
+--------------+--------+-----------+
only showing top 5 rows


Q11
Transformations

Transformations are operations that modify or create a new DataFrame or RDD, such as .filter(), .select(), and .map(). They are lazy, meaning Spark does not execute them immediately. Instead, it records these operations in a DAG (Lineage Graph) and waits until an action is called. This allows Spark to optimize the execution plan before processing the data.

Actions

Actions are operations that trigger the execution of all the transformations stored in the DAG. Examples include .count(), .show(), .collect(), and .write(). Unlike transformations, actions are eager, meaning they execute the job immediately. They either return a result (such as a count, array, or value) to the Driver or write the processed data to external storage.

In [14]:
#Q12
df_superstore.write.mode("overwrite").parquet("superstore_input_parquet")

df_input = spark.read.parquet("superstore_input_parquet")

df_clean = df_input.filter(df_input["Customer ID"].isNotNull())

df_clean.write \
    .option("header", "true") \
    .mode("overwrite") \
    .csv("superstore_output_csv")

Q13
Client Mode

In Client Mode, the Driver runs on the machine where the Spark job is submitted (such as a laptop, edge node, or Jupyter notebook). Since the Driver is outside the cluster, communication with executors happens over the network, which can increase latency. If the client machine disconnects or shuts down, the Spark job also stops. Client mode is best suited for development, testing, debugging, and interactive analysis.

Cluster Mode

In Cluster Mode, the Driver runs inside the cluster on a worker node and is managed by the Cluster Manager (such as YARN, Kubernetes, or Spark Standalone). Because the Driver and Executors are within the same cluster, communication is faster and more efficient. Cluster mode also provides better fault tolerance, as the Cluster Manager can restart the Driver if it fails. It is the preferred mode for production workloads, scheduled jobs, and large-scale batch processing.

In [16]:
#Q14
from pyspark.sql.functions import col

df_superstore_filtered = df_superstore.filter(
    (col("Region") == "Central") | (col("Segment") == "Consumer")
)

df_superstore_filtered.select("Order ID", "Region", "Segment").show(5)

+--------------+------+--------+
|      Order ID|Region| Segment|
+--------------+------+--------+
|CA-2016-152156| South|Consumer|
|CA-2016-152156| South|Consumer|
|US-2015-108966| South|Consumer|
|US-2015-108966| South|Consumer|
|CA-2014-115812|  West|Consumer|
+--------------+------+--------+
only showing top 5 rows


Q15
.collect()

Thisaction retrieves all rows from every executor and brings them into the Driver's memory. While this is fine for small datasets, it becomes dangerous for large datasets because the Driver must store everything in RAM. If the dataset is larger than the Driver's available memory, it can cause an OutOfMemoryError (OOM) and crash the Spark application. Therefore, .collect() should only be used when the result is small.

.show(5)

iits action returns only the first 5 rows of the dataset to the Driver for display. Since only a tiny amount of data is transferred, memory usage remains very low, making it safe even for datasets that are several terabytes or petabytes in size. Spark also optimizes execution by processing only enough data to retrieve those 5 rows. As a result, .show(5) is the preferred method for previewing data and verifying the schema.